# Download Logs — Episode Replay Processing

Loads raw episode JSON files from `11-download_logs/00-raw`, extracts the winner's observations and actions at each timestep, and saves the processed data to `11-download_logs/01-winner`.

In [18]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import math

CENTER_X = 50.0
CENTER_Y = 50.0
MAX_SPEED = 6.0
PLANET_MARGIN = 0.1

# 01 - Save Winner Data

`get_winner_data(data)` finds the winning agent via `np.argmax(data["rewards"])`, then collects each timestep's observation dict and attaches the agent's action to it.

In [2]:
def get_winner_data(data):
    winner_i = np.argmax(data["rewards"])
    nb_timestep = len(data["steps"])
    new_data = []
    for i in range(nb_timestep):
        obs = data["steps"][i][winner_i]["observation"]
        obs["action"] = data["steps"][i][winner_i]["action"]
        new_data.append(obs)
    return new_data

Iterates over every raw episode file, transforms it with `get_winner_data`, and writes the winner's step list to `01-winner/` under the same filename.

In [3]:
raw_dir = Path("11-download_logs/00-raw")

winner_dir = Path("11-download_logs/01-winner")
winner_dir.mkdir(parents=True, exist_ok=True)

for json_file in sorted(raw_dir.glob("*.json")):
    with open(json_file, "r") as f:
        data = json.load(f)
    new_data = get_winner_data(data)
    out_path = winner_dir / json_file.name
    with open(out_path, "w") as f:
        json.dump(new_data, f)
    print(f"Saved: {out_path}")

Saved: 11-download_logs\01-winner\episode-76303112.json
Saved: 11-download_logs\01-winner\episode-76319029.json


## Sanity Check

Verify that in episode `76319029`:
- `bowwowforeach` is the winner
- At game step 41, a fleet was sent from planet 0 with 70 ships
- That fleet was assigned id 32

**Be careful with the step numbers:**
 - the visualiser starts at step 1 whereas the data starts at step 0   
 - the actions displayed at step *t* has been decided at *t-1*
 - the fleet spawn at planet position at *t-1* plus the radius plus a PLANET_MARGIN = 0.1 toward the angle direction

In [4]:
ep_id = "76319029"

with open(f"11-download_logs/00-raw/episode-{ep_id}.json") as f:
    raw = json.load(f)

winner_i = int(np.argmax(raw["rewards"]))
print(f"Winner: {raw['info']['TeamNames'][winner_i]}")

with open(f"11-download_logs/01-winner/episode-{ep_id}.json") as f:
    winner_data = json.load(f)

# Action at game step 42
step_42 = next(obs for obs in winner_data if obs["step"] == 42)
print(f"Action at step 42: {step_42['action']}")
print(f"Planet 0 at step 42: {step_42['planets'][0]}")

# First appearance of fleet 32
print(f"Fleet 32: ")
for obs in winner_data:
    fleet_32 = [f for f in obs["fleets"] if f[0] == 32]
    if fleet_32:
        print(f" {obs["step"]}: {fleet_32}")

Winner: bowwowforeach
Action at step 42: [[0, 5.199833393096924, 70]]
Planet 0 at step 42: [0, 0, 90.0848389632002, 81.13886629663472, 2.6094379124341005, 5, 5]
Fleet 32: 
 42: [[32, 0, 92.95177753556223, 75.73067016848269, 5.199833393096924, 0, 70]]
 43: [[32, 0, 94.5496980949142, 72.71635103883983, 5.199833393096924, 0, 70]]
 44: [[32, 0, 96.14761865426618, 69.70203190919698, 5.199833393096924, 0, 70]]


# 02 - Augment actions with destination planet, discard useless ships


In [85]:
def dataframe_fleets_planets(data):
    fleet_rows = []
    planet_rows = []

    for obs in winner_data:
        step = obs["step"]

        for f in obs["fleets"]:
            fleet_rows.append({
                "step":           step,
                "id":             f[0],
                "owner":          f[1],
                "x":              f[2],
                "y":              f[3],
                "angle":          f[4],
                "from_planet_id": f[5],
                "ships":          f[6],
            })

        for p in obs["planets"]:
            planet_rows.append({
                "step":       step,
                "id":         p[0],
                "owner":      p[1],
                "x":          p[2],
                "y":          p[3],
                "radius":     p[4],
                "ships":      p[5],
                "production": p[6],
            })

    df_fleet   = pd.DataFrame(fleet_rows).drop_duplicates(subset=["step", "id"]).reset_index(drop=True)
    df_planets = pd.DataFrame(planet_rows).drop_duplicates(subset=["step", "id"]).reset_index(drop=True)
    return df_fleet, df_planets


def fleet_speed(nb_ships):
    if nb_ships <= 1:
        return 1.0
    ratio = math.log(nb_ships) / math.log(1000.0)
    return 1.0 + (MAX_SPEED - 1.0) * max(0.0, min(1.0, ratio)) ** 1.5


def augment_data(data):
    df_fleet, df_planets = dataframe_fleets_planets(data)

    df_fleet_last = (
        df_fleet
        .assign(
            first_step = lambda df: df.groupby("id")["step"].transform("min")
        )
        .groupby("id")
        .last()
        .reset_index()
        .assign(
            speed = lambda df: df["ships"].apply(fleet_speed),
            dx = lambda df: df["speed"] * np.cos(df["angle"]),
            dy = lambda df: df["speed"] * np.sin(df["angle"]),
            next_step = lambda df: df["step"] + 1,
        )
        .merge(
            df_planets,
            left_on=["next_step"],
            right_on=["step"],
            suffixes=("_fleet", "_planet"),
            how="inner"
        )
        .assign(
            t = lambda df: ((df["dx"] * (df["x_planet"] - df["x_fleet"]) + df["dy"] * (df["y_planet"] - df["y_fleet"])) / df["speed"] ** 2).clip(lower=0, upper=1),
            distance = lambda df: np.sqrt((df["x_fleet"] + df["dx"] * df["t"] - df["x_planet"]) ** 2 + (df["y_fleet"] + df["dy"] * df["t"] - df["y_planet"]) ** 2),
            on_target = lambda df: df["distance"] < df["radius"] + PLANET_MARGIN,
            angle_3 = lambda df: df["angle"].round(3)
        )
        .query("on_target")
        .groupby("id_fleet")
        .first()
        .set_index(["first_step", "from_planet_id", "angle_3", "ships_fleet"])
    )
    fleet_arrival = df_fleet_last.id_planet.to_dict()

    for step, obs in enumerate(data):
        if step == 0:
            continue
        actions = obs["action"]
        new_actions = []
        for from_planet_id, angle, ships_fleet in actions:
            angle_3 = round(angle, 3)
            tuple_key = (step, from_planet_id, angle_3, ships_fleet)
            if tuple_key in fleet_arrival:
                new_actions.append([from_planet_id, angle, ships_fleet, fleet_arrival[tuple_key]])
            # else:
            #     print("No arrival found for action:", tuple_key)
        data[step-1]["action"] = new_actions
    return data

In [86]:
winner_dir = Path("11-download_logs/01-winner")
winner_dir.mkdir(parents=True, exist_ok=True)

augment_dir = Path("11-download_logs/02-augment")
augment_dir.mkdir(parents=True, exist_ok=True)

for json_file in sorted(winner_dir.glob("*.json")):
    with open(json_file, "r") as f:
        data = json.load(f)
    new_data = augment_data(data)
    out_path = augment_dir / json_file.name
    with open(out_path, "w") as f:
        json.dump(new_data, f)
    print(f"Saved: {out_path}")

Saved: 11-download_logs\02-augment\episode-76303112.json
Saved: 11-download_logs\02-augment\episode-76319029.json


### Sanity Check

Verify that in episode `76319029`:
 - `bowwowforeach` is the winner
 - At game step 41, a fleet was sent from planet 0 with 70 ships
 - That fleet was assigned id 32
 - **to planet 8**

**Be careful with the step numbers:**
 - the visualiser starts at step 1 whereas the data starts at step 0   
 - the actions displayed at step *t* has been decided at *t-1*
 - the fleet spawn at planet position at *t-1* plus the radius plus a PLANET_MARGIN = 0.1 toward the angle direction

In [83]:
ep_id = "76319029"

with open(f"11-download_logs/02-augment/episode-{ep_id}.json") as f:
    augment_data = json.load(f)

# Action at game step 41
step_41 = next(obs for obs in augment_data if obs["step"] == 41)
print(f"Action at step 41: {step_41['action']}")
print(f"Planet 0 at step 41: {step_41['planets'][0]}")

# First appearance of fleet 32
print(f"Fleet 32: ")
for obs in augment_data:
    fleet_32 = [f for f in obs["fleets"] if f[0] == 32]
    if fleet_32:
        print(f" {obs["step"]}: {fleet_32}")

Action at step 41: [[0, 5.199833393096924, 70, 8]]
Planet 0 at step 41: [0, 0, 90.0848389632002, 81.13886629663472, 2.6094379124341005, 70, 5]
Fleet 32: 
 42: [[32, 0, 92.95177753556223, 75.73067016848269, 5.199833393096924, 0, 70]]
 43: [[32, 0, 94.5496980949142, 72.71635103883983, 5.199833393096924, 0, 70]]
 44: [[32, 0, 96.14761865426618, 69.70203190919698, 5.199833393096924, 0, 70]]


## Task 1: Env State Injection Findings

### What was tested
We explored how to inject a saved `obs` dict (from `11-download_logs/02-augment/`) into a fresh `kaggle_environments` orbit_wars env and step it with custom actions.

### env.state structure
- `env.state[0]` is a `kaggle_environments.utils.Struct` (subclass of `dict`)
- `env.state[0]` keys: `['action', 'reward', 'info', 'observation', 'status']`
- `env.state[0].observation` is also a `Struct` with keys:
  `['remainingOverageTime', 'step', 'planets', 'fleets', 'player', 'angular_velocity', 'initial_planets', 'next_fleet_id', 'comets', 'comet_planet_ids']`

### Working injection pattern
**Approach A (canonical):** dict-style key assignment works directly on `Struct`:
```python
for key in OBS_KEYS:
    if key in obs_dict:
        env.state[0].observation[key] = obs_dict[key]
        env.state[1].observation[key] = obs_dict[key]
```

**Approach B:** `setattr` also works (since `Struct.__dict__` holds the data).

Both approaches are confirmed working.

### Critical surprise: step counter reset
After the first `env.step()`, the framework **overwrites** `obs['step']` with `len(env.steps)` (see `kaggle_environments/core.py:602`):
```python
new_state[0].observation.step = 0 if self.done else len(self.steps)
```

So after injecting `step=41` and calling `env.step()`, the resulting `obs['step']` becomes `1`, not `42`. Subsequent steps increment normally: 2, 3, 4...

**However**, the injected `step` value IS used during physics computation inside the interpreter (planet rotation angle calculation at `orbit_wars.py:537,568`), so the physics are correct for the first step. After that, the env's own step counter drives planet rotation — which will be wrong if the game was injected mid-episode. For the purpose of simulating N future steps from a saved state, this means orbital positions will be computed from `step=1` rather than `step=42`, introducing drift on orbiting planets.

### Implication for pipeline/env_sim.py
The injection pattern is:
1. `env = ke.make("orbit_wars", debug=False)` + `env.reset()`
2. Assign all obs keys to both `env.state[0].observation` and `env.state[1].observation`
3. Call `env.step([done_action, []])` with the recorded winner action (opponent action unknown, use `[]`)
4. Call `env.step([[], []])` N more times, collecting `env.state[0].observation['planets']` each time

The `obs['step']` in returned observations will be 1, 2, 3... (not 42, 43, 44...) — consumers should track the true step separately if needed.


In [ ]:
"""
Task 1: Working injection pattern for orbit_wars env state injection.
Confirmed: both dict-style assignment (Approach A) and setattr (Approach B) work.
Canonical pattern below uses Approach A.
"""
import kaggle_environments as ke
import json
import warnings, logging
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

# Keys present in env.state[0].observation after env.reset()
OBS_KEYS = [
    'remainingOverageTime', 'step', 'planets', 'fleets', 'player',
    'angular_velocity', 'initial_planets', 'next_fleet_id', 'comets', 'comet_planet_ids'
]

def make_env_from_obs(obs_dict, done_action=None, n_future_steps=5):
    """
    Inject a saved obs dict into a fresh orbit_wars env and step it forward.

    Parameters
    ----------
    obs_dict : dict
        Saved observation with keys: step, planets, fleets, angular_velocity, etc.
    done_action : list or None
        The 'done actions' (fleets to send this turn) — i.e. obs_dict['action'].
        Pass [] or None to skip.
    n_future_steps : int
        How many additional empty steps to simulate after the done_action step.

    Returns
    -------
    list of list
        Planet snapshots at each future step. Each entry is a copy of obs['planets'].
    """
    env = ke.make("orbit_wars", debug=False)
    env.reset()

    # Inject all obs keys into both player observations
    for key in OBS_KEYS:
        if key in obs_dict:
            env.state[0].observation[key] = obs_dict[key]
            env.state[1].observation[key] = obs_dict[key]

    # Step 1: apply done actions (winner's action; opponent unknown so use [])
    action0 = done_action if done_action else []
    env.step([action0, []])

    # Steps 2..N+1: empty actions, collect planet snapshots
    planet_snapshots = []
    for _ in range(n_future_steps):
        env.step([[], []])
        planets = env.state[0].observation['planets']
        planet_snapshots.append([p[:] for p in planets])  # deep copy

    return planet_snapshots


# --- Demo: inject obs_41 and simulate 5 future steps ---
with open("11-download_logs/02-augment/episode-76319029.json") as f:
    ep_data = json.load(f)

obs_41 = next(o for o in ep_data if o["step"] == 41)
done_action = obs_41.get("action", [])

snapshots = make_env_from_obs(obs_41, done_action=done_action, n_future_steps=5)

print(f"Injected obs at step={obs_41['step']}, planets={len(obs_41['planets'])}, fleets={len(obs_41['fleets'])}")
print(f"done_action: {done_action}")
print(f"\nPlanet[0] (id={obs_41['planets'][0][0]}, owner={obs_41['planets'][0][1]}, prod={obs_41['planets'][0][6]}) ships over next 5 steps:")
for i, snap in enumerate(snapshots):
    print(f"  future step {i+1}: ships={snap[0][5]}")

print(f"\nStructure confirmed:")
print(f"  env.state[0] type: kaggle_environments.utils.Struct (subclass of dict)")
print(f"  obs keys: {OBS_KEYS}")
print(f"  obs['step'] resets to 1 after first env.step() — track true step separately")


# 03 - Combinatorial Augmentation

For each obs, generate all (done_set, action_to_do) pairs plus a stop row.
This expands the dataset to train the model on "given actions already sent, predict next action".
Output: `11-download_logs/03-combinations/*.json`

In [ ]:
from itertools import combinations as _combinations
import copy

def generate_combinations(obs):
    """
    Expands one obs into all (done_set, action_to_do) pairs plus a stop row.
    obs["action"] = [[from_planet_id, angle, ships, dest_planet_id], ...]
    Returns list of dicts: {obs, done_set, action_to_do}.
    """
    actions = obs["action"]
    nb_action = len(actions)
    rows = []
    for nb_done in range(nb_action):
        for done_indices in _combinations(range(nb_action), nb_done):
            done = [actions[i] for i in done_indices]
            remaining = [actions[i] for i in range(nb_action) if i not in done_indices]
            for action_to_do in remaining:
                rows.append({
                    "obs": obs,
                    "done_set": done,
                    "action_to_do": action_to_do,
                })
    rows.append({"obs": obs, "done_set": actions, "action_to_do": None})
    return rows

# Test: 2 actions → 5 rows: (∅,A),(∅,B),([A],B),([B],A),(stop)
mock_obs = {
    "step": 41,
    "action": [[0, 1.0, 70, 8], [1, 2.0, 30, 5]],
    "planets": [],
    "fleets": [],
}
combos = generate_combinations(mock_obs)
assert len(combos) == 5, f"Expected 5, got {len(combos)}"
assert combos[-1]["action_to_do"] is None
assert combos[0]["done_set"] == []
assert combos[2]["done_set"] == [[0, 1.0, 70, 8]]
assert combos[2]["action_to_do"] == [1, 2.0, 30, 5]
print("generate_combinations OK")

In [ ]:
augment_dir = Path("11-download_logs/02-augment")
combo_dir   = Path("11-download_logs/03-combinations")
combo_dir.mkdir(parents=True, exist_ok=True)

for json_file in sorted(augment_dir.glob("*.json")):
    with open(json_file) as f:
        data = json.load(f)
    all_rows = []
    for obs in data:
        if not obs.get("action"):
            continue
        all_rows.extend(copy.deepcopy(generate_combinations(obs)))
    out_path = combo_dir / json_file.name
    with open(out_path, "w") as f:
        json.dump(all_rows, f)
    print(f"Saved {len(all_rows):,} rows → {out_path}")

# 04 - Env Simulation

For each augmented row, inject obs into the env, apply done_set, then step 10 times.
Records future planet states for use as ML features.
Output: `11-download_logs/04-simulate/*.json`
NB_FUTURE_STEP = 10 — re-run this step only to change it.

In [ ]:
import sys
sys.path.insert(0, ".")
from pipeline.env_sim import simulate_futures

NB_FUTURE_STEP = 10

with open("11-download_logs/03-combinations/episode-76319029.json") as f:
    combos = json.load(f)

row0 = combos[0]
snaps = simulate_futures(row0["obs"], row0["done_set"], NB_FUTURE_STEP)
assert len(snaps) == NB_FUTURE_STEP
assert len(snaps[0]) > 0
print(f"Sanity OK — {NB_FUTURE_STEP} snapshots, {len(snaps[0])} planets each")
print(f"  done_set: {row0['done_set']}")
print(f"  action_to_do: {row0['action_to_do']}")

In [ ]:
combo_dir    = Path("11-download_logs/03-combinations")
simulate_dir = Path("11-download_logs/04-simulate")
simulate_dir.mkdir(parents=True, exist_ok=True)

for json_file in sorted(combo_dir.glob("*.json")):
    with open(json_file) as f:
        rows = json.load(f)
    for row in rows:
        row["future_planets"] = simulate_futures(
            row["obs"], row["done_set"], NB_FUTURE_STEP
        )
    out_path = simulate_dir / json_file.name
    with open(out_path, "w") as f:
        json.dump(rows, f)
    print(f"Saved {len(rows):,} rows → {out_path}")